# Chapter 4: Multi-Input and Multi-Output Models
**Module 02 – Intermediate Deep Learning with PyTorch**

> *Instructor: Michal Oleszak, Machine Learning Engineer*

## 4.1 Why Multi-Input?

Sometimes a single data stream isn't enough. Multi-input models let you:
- **Use more information** from different sources
- Build **multi-modal models** (e.g., image + text)
- Enable **metric learning** (learn similarity between samples)
- Support **self-supervised learning**

### The Omniglot Dataset
A character recognition dataset with 1,623 different handwritten characters from 50 alphabets. Used for few-shot learning — can the model recognize characters from only a few examples?

## 4.2 Tensor Concatenation — The Core Operation

In [ ]:
import torch

# Concatenate along feature dimension (dim=1)
x = torch.tensor([[1, 2, 3]])
y = torch.tensor([[4, 5, 6]])

combined = torch.cat([x, y], dim=1)
print("x:", x)
print("y:", y)
print("Concatenated:", combined)  # [[1, 2, 3, 4, 5, 6]]

## 4.3 Two-Input Dataset

In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np

class OmniglotDataset(Dataset):
    """Dataset returning (image, alphabet_one_hot, label) triples."""

    def __init__(self, transform, samples):
        """
        Args:
            transform: image transformations
            samples: list of (img_path, alphabet_vector, label)
        """
        self.transform = transform
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, alphabet, label = self.samples[idx]
        # Load image in grayscale
        img = Image.open(img_path).convert('L')
        img = self.transform(img)
        return img, alphabet, label


# Example sample structure:
example_sample = [
    ('path/to/char.png',
     np.array([1., 0., 0., 0.]),  # one-hot encoded alphabet
     0)                            # label
]
print("Dataset structure defined.")

## 4.4 Two-Input Neural Network

In [ ]:
import torch.nn as nn

class MultiInputNet(nn.Module):
    """
    Combines image features with alphabet metadata.
    Input 1: image tensor (1, H, W)
    Input 2: alphabet one-hot vector
    """
    def __init__(self, num_alphabets=30, num_classes=100):
        super(MultiInputNet, self).__init__()

        # Image branch: CNN
        self.image_branch = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),  # Assumes 28x28 input
            nn.ReLU()
        )

        # Alphabet branch: simple linear
        self.alphabet_branch = nn.Sequential(
            nn.Linear(num_alphabets, 32),
            nn.ReLU()
        )

        # Combined classifier
        self.classifier = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, img, alphabet):
        img_features = self.image_branch(img)
        alph_features = self.alphabet_branch(alphabet)

        # Concatenate both feature vectors
        combined = torch.cat([img_features, alph_features], dim=1)
        output = self.classifier(combined)
        return output

# Test the model
model = MultiInputNet(num_alphabets=30, num_classes=100)
img_batch = torch.randn(8, 1, 28, 28)           # 8 grayscale 28x28 images
alph_batch = torch.randn(8, 30)                  # 8 alphabet vectors
out = model(img_batch, alph_batch)
print(f"Output shape: {out.shape}")  # (8, 100)
print(model)

## 4.5 Training a Multi-Input Model

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training loop sketch (with real DataLoader these would be loaded from files)
for epoch in range(3):
    model.train()
    # Simulate a batch
    img_batch  = torch.randn(16, 1, 28, 28)
    alph_batch = torch.randn(16, 30)
    labels     = torch.randint(0, 100, (16,))

    outputs = model(img_batch, alph_batch)
    loss = criterion(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/3 | Loss: {loss.item():.4f}")

## Summary

| Pattern | Use Case |
|---|---|
| Multi-input | Combine image + metadata, two related images, multimodal data |
| `torch.cat()` | Merge feature vectors from different branches |
| Separate branches | Process each input type with the right architecture |